# 롤 API 데이터 수집하기

In [43]:
import requests
import json
import time
import numpy as np
import pandas as pd

## API를 통한 소환사 정보 수집

In [44]:
# Lol Api에서 소환사 명을 통해서 게임 정보를 알 수 있는 puuid 불러오기 
summoner_Name = 'one summer drive' #소환사명
api_key = "RGAPI-bd19aa3d-2669-45a7-a538-cf8f1baa0113" # api key
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Charset": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://developer.riotgames.com",
    "X-Riot-Token": "RGAPI-bd19aa3d-2669-45a7-a538-cf8f1baa0113"
}

summoner_url="https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key

In [45]:
# requests를 통해서 puuid 추출하기
def summoner(summoner_Name, api_key):
    summoner_url = "https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key
    return requests.get(summoner_url, headers=request_headers).json()['puuid']
summoner_puuid = summoner(summoner_Name, api_key)
summoner_puuid

'cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72J0XfluBWZsNURa-_rKA5AR_vMndVW8tg'

In [46]:
# match id를 확인할 수 있는 api를 통해서 조회하기
# match_count : 조회하고 싶은 매치의 수
# 더 많이 조회를 하고 싶으나 횟수 제한이 있다. (100) 이번 시즌 내가 플레이한 62게임에 대해서 조사하기 
def match(summoner_puuid, match_count):
    match_url = "https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/"+summoner_puuid+"/ids?start=0&count="+str(match_count)
    return requests.get(match_url, headers=request_headers).json()
match_id = match(summoner_puuid, 90)
match_id

['KR_6024617209',
 'KR_6024522598',
 'KR_6024290118',
 'KR_6024206432',
 'KR_6018696284',
 'KR_6018287979',
 'KR_6017207879',
 'KR_6017033024',
 'KR_6016828310',
 'KR_6016772762',
 'KR_6016770544',
 'KR_6016651859',
 'KR_6016575318',
 'KR_6015787618',
 'KR_6015488822',
 'KR_6015088138',
 'KR_6014287649',
 'KR_6012864614',
 'KR_6012476838',
 'KR_6011712595',
 'KR_6011597486',
 'KR_6010856241',
 'KR_6009908963',
 'KR_6009876008',
 'KR_6009892943',
 'KR_6009828517',
 'KR_6009775487',
 'KR_6009753833',
 'KR_6008388001',
 'KR_6008101825',
 'KR_6007718428',
 'KR_6007632404',
 'KR_6007543140',
 'KR_6007407898',
 'KR_6007030534',
 'KR_6006751091',
 'KR_6006310180',
 'KR_6006082114',
 'KR_6005589922',
 'KR_6005521617',
 'KR_6005292412',
 'KR_6004906534',
 'KR_6002949666',
 'KR_6002335329',
 'KR_6000663825',
 'KR_6000095173',
 'KR_5995456456',
 'KR_5993880535',
 'KR_5992325428',
 'KR_5992263876',
 'KR_5992247758',
 'KR_5992174929',
 'KR_5992139509',
 'KR_5992067375',
 'KR_5990749357',
 'KR_59902

In [47]:
# 게임 세부내용 불러오기
# 조회는 하나씩만 가능하므로 for문이나 while문을 돌려 하나씩 조회한다.
def game(match_id_num):
    game_url = "https://asia.api.riotgames.com/lol/match/v5/matches/"+match_id_num
    return requests.get(game_url, headers=request_headers).json()['info']['participants']


## 분석에 사용할 90개의 데이터 수집하여 데이터 프레임 생성
- 100개로 하고 돌렸으나 API응답문제가 발생하였다

In [48]:
#Future Warning 메세지 제거하기
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# 게임 데이터를 담을 DataFrame 생성하기
game_df = pd.DataFrame(columns=['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp', 'summoner_spell1', 'summoner_spell2' ,
                                'kills', 'deaths', 'assists', 'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin', 'teamDamagePer(%)',
                                'killParticipation(%)', 'totalDamageTaken', 'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
                                'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent', 'gameEndedInEarlySurren', 'gameEndedInSurren',
                                'teamEarlySurrn', 'timePlayed'])

for i in range(len(match_id)):
    print(i)
    game_content = game(match_id[i])
    for j in range(len(game_content)):
        try:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':float(game_content[j]['challenges']['maxCsAdvantageOnLaneOpponent']),
                'maxLevelLeadLaneOpponent':float(game_content[j]['challenges']['maxLevelLeadLaneOpponent']),
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True)
        except:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':0,
                'maxLevelLeadLaneOpponent':0,
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True) 

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89


In [49]:
game_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   puuid                         900 non-null    object 
 1   teamPosition                  900 non-null    object 
 2   individualPosition            900 non-null    object 
 3   champName                     900 non-null    object 
 4   champExp                      900 non-null    float64
 5   summoner_spell1               900 non-null    float64
 6   summoner_spell2               900 non-null    float64
 7   kills                         900 non-null    float64
 8   deaths                        900 non-null    float64
 9   assists                       900 non-null    float64
 10  damageDealtToBuildings        900 non-null    float64
 11  totalDamageDealtToChamp       900 non-null    float64
 12  damagePerMin                  900 non-null    float64
 13  teamD

In [50]:
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,18073.0,4.0,12.0,10.0,4.0,5.0,...,21.0,15723.0,438.402459,215.0,57.00,2.0,False,True,False,3585.0
1,fTgvC1ZjaMvTWlQ4_97W0h931BNv01xIGLe23PLgJlTGBo...,JUNGLE,JUNGLE,Vi,18132.0,11.0,4.0,5.0,6.0,8.0,...,20.0,13931.0,388.428870,228.0,83.75,3.0,False,True,False,3585.0
2,RNnTONbpneCNdnFE-TwgFR0LWhtD_xiQ5gMPpGq0l8J-xa...,MIDDLE,MIDDLE,Irelia,18291.0,14.0,4.0,11.0,6.0,7.0,...,29.0,17304.0,482.465807,262.0,55.00,1.0,False,True,False,3585.0
3,Ht7MsUXrZx1F7zv9J5ouSx3erSvDHfHv6kFQQ2NLo25ucL...,BOTTOM,BOTTOM,MissFortune,14546.0,7.0,4.0,1.0,6.0,6.0,...,16.0,9798.0,273.196957,156.0,0.00,0.0,False,True,False,3585.0
4,9fFmEswTeX5fHm5a4S4VZhbIMhIsvtCpSU26nPiKARJHIo...,UTILITY,UTILITY,Morgana,12969.0,4.0,14.0,1.0,5.0,9.0,...,15.0,9566.0,266.720296,61.0,28.00,1.0,False,True,False,3585.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,8UYh2JGoEUP4VKuJor8GopmoSvwTs9JyUukE15vXL52A_M...,,Invalid,Nasus,14776.0,32.0,4.0,12.0,10.0,11.0,...,27.0,13124.0,772.707227,42.0,5.00,1.0,False,False,False,1698.0
896,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,,Invalid,Ornn,14033.0,4.0,32.0,4.0,11.0,16.0,...,19.0,10315.0,607.365632,40.0,9.00,0.0,False,False,False,1698.0
897,wBdZ_BD1xlwH71ZI0tvZTo2nrii4sEpShcvm0L5wnaFOIn...,,Invalid,Shen,14352.0,32.0,4.0,6.0,11.0,14.0,...,20.0,10643.0,626.641100,15.0,5.00,0.0,False,False,False,1698.0
898,GKzAJA_Z9n3n5SaqP0x8LoMefMw-sonZZGTniu-bzppHTE...,,Invalid,Katarina,14005.0,32.0,4.0,7.0,11.0,15.0,...,18.0,10883.0,640.760383,36.0,5.00,0.0,False,False,False,1698.0


## 가설 확인하기

```
인게임 속의 트롤링은 처음부터 발생할 수도 있고 중간에 소환사의 변심으로 인해서 발생할 수도 있고 수 많은 경우로 인해 발생한다.
기본적으로 의도적인 죽음, 다른 소환사들 방해, 한타에 합류를 하지 않거나, 우물에서 잠수를 하거나 
위와 같은 행위들은 기본적으로 DPM의 하락을 가져온다. 또한 안정적으로 성장을 한 상대 라이너에 비해서 성장 측면에서 많은 차이가 나타난다. 

픽창에서의 싸움으로 인한 소환사 스펠, 주 포지션이 아니므로 인한 트롤 등에 대해서는 소환사 스펠을 통해서 확인한다. 
```

- 가장 최근 게임의 트롤은 워익이었다. 
- 욕설과 함께 한타참여X, 정글링만 하거나, 우물에서 잠수 
- 딜량에 대해서 수치화 하기 
- 게임은 빠른 서렌이 나오는 15분부터 끝나기 시작한다.
- 15분까지는 기본적으로 라인전이 끝나는 시기, 탑, 미드, 바텀의 경우에는 라인전을 통해서 딜량을 쌓을수 있으나 정글의 경우에는 힘들다. 
- 게임이 장기간으로 가는경우 서포터는 딜량이 작기 마련이다. 
- 원딜의 경우에는 코어가 뜨면 뜰 수록 강해진다. 
- 위와 같은 경우를 수치화 조정해서 딜량을 계산한다.

## Match에 포함되어 있는 칼바람, 일반게임 정보 제거하기 

In [51]:
# 포지션에 Top, Jungle, Middle, Botoom, Utility 이외의 공백인 행들이 일반게임, 칼바람나락의 결과
game_df['teamPosition'].unique()

array(['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY', ''], dtype=object)

In [52]:
game_df['teamPosition']==''.index

0      False
1      False
2      False
3      False
4      False
       ...  
895    False
896    False
897    False
898    False
899    False
Name: teamPosition, Length: 900, dtype: bool

In [53]:
# 포지션 선택이 없는 게임 제거하기
# 29게임이 제외되었다. 
game_df = game_df.drop(game_df[game_df['teamPosition']==''].index, axis=0).reset_index(drop=True)
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,18073.0,4.0,12.0,10.0,4.0,5.0,...,21.0,15723.0,438.402459,215.0,57.00,2.0,False,True,False,3585.0
1,fTgvC1ZjaMvTWlQ4_97W0h931BNv01xIGLe23PLgJlTGBo...,JUNGLE,JUNGLE,Vi,18132.0,11.0,4.0,5.0,6.0,8.0,...,20.0,13931.0,388.428870,228.0,83.75,3.0,False,True,False,3585.0
2,RNnTONbpneCNdnFE-TwgFR0LWhtD_xiQ5gMPpGq0l8J-xa...,MIDDLE,MIDDLE,Irelia,18291.0,14.0,4.0,11.0,6.0,7.0,...,29.0,17304.0,482.465807,262.0,55.00,1.0,False,True,False,3585.0
3,Ht7MsUXrZx1F7zv9J5ouSx3erSvDHfHv6kFQQ2NLo25ucL...,BOTTOM,BOTTOM,MissFortune,14546.0,7.0,4.0,1.0,6.0,6.0,...,16.0,9798.0,273.196957,156.0,0.00,0.0,False,True,False,3585.0
4,9fFmEswTeX5fHm5a4S4VZhbIMhIsvtCpSU26nPiKARJHIo...,UTILITY,UTILITY,Morgana,12969.0,4.0,14.0,1.0,5.0,9.0,...,15.0,9566.0,266.720296,61.0,28.00,1.0,False,True,False,3585.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,izkdkzth-nRWiGU1ssUGwGOZSmIx3E0CnVw8TvmCdbADRI...,TOP,TOP,Lucian,10465.0,12.0,4.0,5.0,2.0,3.0,...,21.0,9242.0,512.974362,160.0,63.00,2.0,False,True,False,1800.0
656,GLUb-YiZWPONWpR3wd4E7w17Qmez0-Zowkk-ZmbdchnAAL...,JUNGLE,JUNGLE,Viego,8087.0,11.0,4.0,5.0,4.0,3.0,...,35.0,7629.0,423.453881,97.0,28.00,2.0,False,True,False,1800.0
657,QOU8VObhOsaqbkB3fo8RhBWfhSnBlA6eL6LN5VMP3rl9I3...,MIDDLE,MIDDLE,Lissandra,9822.0,12.0,4.0,7.0,1.0,3.0,...,20.0,8856.0,491.572199,133.0,44.00,2.0,False,True,False,1800.0
658,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,6736.0,4.0,7.0,1.0,1.0,6.0,...,10.0,7273.0,403.683500,147.0,35.00,1.0,False,True,False,1800.0


### Play시간대 별로 나누기 

#### 20분 이내 종료된 게임

In [54]:
# 5게임만이 61게임중에서 빠르게 끝난 게임 
game_played20down = game_df.copy()[game_df['timePlayed']<2000].reset_index(drop=True)
game_played20down

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,B6f-hh1my-rb1rAAAQTgeJGKVcSXZxZpVPRdWWsUvVeVDO...,TOP,TOP,Nilah,6025.0,4.0,12.0,1.0,3.0,0.0,...,29.0,4026.0,296.781612,88.0,7.00,1.0,False,True,False,1355.0
1,ciPKyG5R0qSB1XvlPWNYaIsnyWDdiBH6Gf6Z2QFw7K1feU...,JUNGLE,JUNGLE,MasterYi,3038.0,11.0,4.0,0.0,7.0,1.0,...,33.0,3081.0,227.123002,35.0,0.00,0.0,False,True,False,1355.0
2,bVKmCzSI3ZRvfE4OZO-IBJ7zol5FjupNBH6kQVmzrH101X...,MIDDLE,MIDDLE,Zyra,6419.0,4.0,12.0,0.0,0.0,0.0,...,5.0,4479.0,330.173516,105.0,7.00,1.0,False,True,False,1355.0
3,Wufvx6Ms2y_bj-zgjjUV4wvOBFU0wpMGg10RRyMQ-7pgd2...,BOTTOM,BOTTOM,Lucian,3766.0,7.0,4.0,1.0,4.0,0.0,...,20.0,4586.0,338.060788,93.0,11.00,0.0,False,True,False,1355.0
4,-dUAnoFX34YOzyqatXBEfUttJfkvkzVK2TbXpQBvs6Y-2A...,UTILITY,UTILITY,Yuumi,3388.0,14.0,3.0,0.0,3.0,1.0,...,13.0,2442.0,180.071371,1.0,0.00,0.0,False,True,False,1355.0
5,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,7597.0,4.0,12.0,3.0,1.0,1.0,...,24.0,5326.0,392.643939,96.0,8.00,2.0,False,True,False,1355.0
6,Dj8_KHWZSrVRcsIMYuLd7Ok9mttISfUy3SlgmovTteTnlV...,JUNGLE,JUNGLE,Amumu,5423.0,11.0,4.0,5.0,1.0,2.0,...,33.0,5272.0,388.655133,84.0,49.00,3.0,False,True,False,1355.0
7,aUu1CsABRqua4d3Wwmh9-cg8w5PLE6u6Y8PG1Wxrf1kzOm...,MIDDLE,MIDDLE,Sylas,7465.0,14.0,4.0,3.0,0.0,0.0,...,21.0,4697.0,346.263318,103.0,7.00,2.0,False,True,False,1355.0
8,HL6hmznifOEV6msIuafQIIrzTtsdJRPCE4KZqck84-UAXP...,BOTTOM,BOTTOM,Jhin,4539.0,7.0,4.0,4.0,0.0,4.0,...,11.0,6170.0,454.865832,98.0,12.00,2.0,False,True,False,1355.0
9,G9HobrTn_m98oKGZeEgjrw2uQbp_HBAc41OsG-RVa6uj7O...,UTILITY,UTILITY,FiddleSticks,4386.0,3.0,4.0,2.0,0.0,5.0,...,11.0,4655.0,343.178177,7.0,5.75,1.0,False,True,False,1355.0


#### 20분 ~ 30분 사이 종료된 게임

In [55]:
# 20분에서 30분 사이 종료된 게임으로는 33게임이 존재한다. 
game_played2030 = game_df.copy()[(2000<= game_df['timePlayed']) & (game_df['timePlayed'] < 3000)].reset_index(drop=True)
game_played2030

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,Y_Goo1Pyrbfda3G7UBm50nKsrIqb4izeJg8ZyfNFO03edc...,TOP,TOP,Sejuani,10612.0,12.0,14.0,2.0,3.0,6.0,...,19.0,6897.0,303.496485,121.0,3.0,1.0,False,False,False,2272.0
1,CMwcORn3va6QDUCuWIdAFVOhl_oMV3aR-BsVQXSGpGMZm5...,JUNGLE,JUNGLE,Warwick,7978.0,4.0,11.0,4.0,13.0,4.0,...,31.0,6834.0,300.724322,68.0,8.0,1.0,False,False,False,2272.0
2,F_mnm6jdpyC7cajw2hZYLK0TmNSF_TjkwEx3NLTC2TlFdG...,MIDDLE,MIDDLE,Lissandra,9813.0,4.0,12.0,4.0,6.0,6.0,...,21.0,8514.0,374.659403,129.0,6.0,1.0,False,False,False,2272.0
3,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,9936.0,4.0,7.0,4.0,3.0,5.0,...,9.0,8061.0,354.730589,141.0,2.0,1.0,False,False,False,2272.0
4,G43ysCfnO1sJOQwfmEKC102gnUbx1CNCyhgSURtgM0z1_8...,UTILITY,UTILITY,Renata,6704.0,4.0,14.0,1.0,8.0,5.0,...,20.0,5834.0,256.743482,38.0,17.0,1.0,False,False,False,2272.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,QxffUyKLIHztu5DPi06OglE9Sf8Gr2NdsNY1d7EhFRfJpe...,TOP,TOP,LeeSin,7324.0,12.0,4.0,3.0,8.0,3.0,...,22.0,6656.0,317.423426,80.0,17.0,1.0,False,True,False,2097.0
346,7laSMnqFaZQgu3zqX_GVtjihk_AykGaA-0xIwjOB_20C1G...,JUNGLE,JUNGLE,Zac,5613.0,11.0,4.0,1.0,9.0,2.0,...,29.0,5155.0,245.845942,66.0,5.0,1.0,False,True,False,2097.0
347,MmWFm-YgIIxp0naNdqhU4hMmW00aI9V8dVM5mCQDdXsipx...,MIDDLE,MIDDLE,Yasuo,9404.0,14.0,4.0,6.0,6.0,4.0,...,23.0,10991.0,524.114813,151.0,65.0,1.0,False,True,False,2097.0
348,tPblUi0-8C2VmSudF5TeoPjvU2ifuXGZfYDHQHGDiirJ7L...,BOTTOM,UTILITY,Jhin,6091.0,7.0,4.0,1.0,8.0,3.0,...,15.0,5339.0,254.619547,70.0,40.0,1.0,False,True,False,2097.0


#### 30분 ~ 40분 사이 종료된 게임

In [56]:
# 20게임이 존재한다.
game_played3040 = game_df.copy()[(3000<= game_df['timePlayed']) & (game_df['timePlayed'] < 4000)].reset_index(drop=True)
game_played3040

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,18073.0,4.0,12.0,10.0,4.0,5.0,...,21.0,15723.0,438.402459,215.0,57.00,2.0,False,True,False,3585.0
1,fTgvC1ZjaMvTWlQ4_97W0h931BNv01xIGLe23PLgJlTGBo...,JUNGLE,JUNGLE,Vi,18132.0,11.0,4.0,5.0,6.0,8.0,...,20.0,13931.0,388.428870,228.0,83.75,3.0,False,True,False,3585.0
2,RNnTONbpneCNdnFE-TwgFR0LWhtD_xiQ5gMPpGq0l8J-xa...,MIDDLE,MIDDLE,Irelia,18291.0,14.0,4.0,11.0,6.0,7.0,...,29.0,17304.0,482.465807,262.0,55.00,1.0,False,True,False,3585.0
3,Ht7MsUXrZx1F7zv9J5ouSx3erSvDHfHv6kFQQ2NLo25ucL...,BOTTOM,BOTTOM,MissFortune,14546.0,7.0,4.0,1.0,6.0,6.0,...,16.0,9798.0,273.196957,156.0,0.00,0.0,False,True,False,3585.0
4,9fFmEswTeX5fHm5a4S4VZhbIMhIsvtCpSU26nPiKARJHIo...,UTILITY,UTILITY,Morgana,12969.0,4.0,14.0,1.0,5.0,9.0,...,15.0,9566.0,266.720296,61.0,28.00,1.0,False,True,False,3585.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Kayle,15748.0,4.0,12.0,2.0,2.0,10.0,...,16.0,11278.0,340.080333,190.0,1.00,0.0,False,True,False,3315.0
216,3QjhP6LhMEM98jxz_bbQejmz1jW1kWoUc9Bgk-nJ-zRTD8...,JUNGLE,JUNGLE,MonkeyKing,18282.0,11.0,4.0,6.0,1.0,5.0,...,25.0,12889.0,388.663675,214.0,52.35,3.0,False,True,False,3315.0
217,w23nDK6_Wg3SJuCz9dtfsUKjzIxm9nh-LRfA_KVLpcmL-K...,MIDDLE,MIDDLE,Irelia,18111.0,14.0,4.0,13.0,9.0,4.0,...,30.0,16285.0,491.070299,255.0,5.25,2.0,False,True,False,3315.0
218,Ggajs4IGPc6MJFU9Lvh1Jn7X6l9yc0ewRCmME2isd5qMlZ...,BOTTOM,BOTTOM,Ezreal,14056.0,7.0,4.0,7.0,3.0,8.0,...,15.0,13485.0,406.644434,229.0,38.00,1.0,False,True,False,3315.0


#### 40분 이후에 종료된 게임

In [57]:
# 3게임이 존재한다. 
game_played40up = game_df.copy()[4000 < game_df['timePlayed']].reset_index(drop=True)
game_played40up

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,IlqYon3McqGJG0th5WG1bYTFyUPDUIMlScWszvvMqQt7Qd...,TOP,TOP,Poppy,25408.0,12.0,4.0,8.0,5.0,7.0,...,21.0,18877.0,391.931189,307.0,77.00,3.0,False,True,False,4815.0
1,1Ki4PlEmbDX2h5__VdSYZA5tXea8oJbbbq3mLcbJWTjVGa...,JUNGLE,JUNGLE,Janna,19322.0,11.0,4.0,1.0,15.0,22.0,...,20.0,12124.0,251.734976,98.0,4.00,1.0,False,True,False,4815.0
2,3twz4G44Wj1kWZMPLSrDeGxCVp8ARFprdFiEAPxNK6o0zk...,MIDDLE,MIDDLE,Akali,22563.0,14.0,4.0,26.0,9.0,2.0,...,24.0,22486.0,466.869134,235.0,26.00,1.0,False,True,False,4815.0
3,FUwkQVEcZbRUb7d_uiY9nBHK4QPca6LBO1Io4DH9Ua2icm...,BOTTOM,BOTTOM,Jinx,24119.0,7.0,4.0,8.0,8.0,16.0,...,21.0,19669.0,408.379721,370.0,150.00,2.0,False,True,False,4815.0
4,ihATf3I8tbukfaBkTGtNcQfQ2_0XzVGOSIWojWwTsjUHDL...,UTILITY,UTILITY,Xerath,20470.0,14.0,4.0,9.0,6.0,11.0,...,14.0,16612.0,344.908673,103.0,40.00,2.0,False,True,False,4815.0
5,Gnj1h1FYYZIqGMaPwKoetO_jbKnUvUvukKaNXUsz9uTHV8...,TOP,TOP,Jax,21418.0,14.0,4.0,7.0,13.0,6.0,...,23.0,16747.0,347.702385,253.0,20.00,1.0,False,True,False,4815.0
6,Dfl5W0HF1Y93K-sRuIeVV6pH3r7eq8zqIVB2UJ63N8CDrf...,JUNGLE,JUNGLE,Viego,20617.0,11.0,4.0,9.0,7.0,15.0,...,25.0,17344.0,360.108521,186.0,89.00,3.0,False,True,False,4815.0
7,bvZq5jVAvK8_TY9gKPGSKx2mXzFBVZqGfrs1IuqWvMcLZt...,MIDDLE,MIDDLE,Ziggs,23850.0,7.0,4.0,6.0,10.0,16.0,...,19.0,19079.0,396.125894,328.0,96.50,2.0,False,True,False,4815.0
8,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,18809.0,4.0,7.0,18.0,9.0,6.0,...,17.0,20659.0,428.942018,220.0,12.00,2.0,False,True,False,4815.0
9,8SIB8WFW_M_an1ElcXOEkw4u4gMcURUMkSusKBw12l4AUb...,UTILITY,UTILITY,Senna,19423.0,14.0,4.0,3.0,13.0,21.0,...,17.0,13826.0,287.058017,73.0,6.00,2.0,False,True,False,4815.0


### 종료된 시간 대별, 라인별 평균 딜량, 챔피언 경험치, 골드획득을 확인한다. 

#### 20분 이내 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인

In [58]:
# 물론 트롤들과, 부캐, 대리, 이상치들이 섞인 평균이나 표본자체가 적어서 그대로 진행하였다. 
# 초반의 경우에는 템이 나오기전의 원딜, 라인전을 하지 않는 정글, 서포터 챔프의 특성으로 인해 
# 탑이나 미드라인에 비해서 딜량이 적게 나오는 것을 확인할 수 있다. 
# 챔피언 경험치 또한 2명이서 같이 받아먹는 봇라인의경우 적게 나오며 탑, 미드 라이너들의 솔로라인 경험치가 높다. 
# 골드획득의 경우에는 서폿을 제외한 모든 라인이 비슷한 것을 확인한다. 
# 20분 이전의 champExp 계산시 BOTTOM라인은 110% Middle라인은 90%, TOP 85%, UTILITY 130%
# 20분 이전의 totalDamageDealtToCHamp 계산시 JUNGLE 105%, MIDDLE 75%, TOP 90%, UTILITY 155%
# 20분 이전의 goldEarned 계산시 UTILITY 135%
# 위와 같은 비율로 평균화를 해주기로 하였다. 
game_played20down.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,5996.083333,6219.083333,6452.500000
JUNGLE,6655.333333,5626.166667,6138.166667
MIDDLE,7583.500000,8134.833333,6058.000000
TOP,7956.916667,7358.666667,6073.416667
UTILITY,5018.333333,3882.250000,4435.916667


#### 20분 이상, 30분 이내 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인

In [59]:
# 20분 이상 종료된 게임에서는 라인전 단계가 끝나고 한타, 오브젝트 싸움, 타워등에 있어서 경험치, 골드의 격차가 더 커질 것이다
# 20분에서 30분이하의 게임에서도 20분과 같이 여전히 바텀과 서폿의 경험치는 상체 라인에 비해서 적은걸로 확인되었다.
# 20분 이상 30분 이내 게임에서 champExp Bottom 110%, UTILIY 125%
# totalDamageDealtToChamp의 경우에는 정글의 라인전 갱킹, 등 여러한 부분에서 떨어지므로 딜량이 적게 나온다. 서폿또한 마찬가지이다.
# JUNGLE 105%, UTILITY 140% 
# goldEarned 에서는 UTILITY의 부분만 조정을 하면 된다. 130%
game_played2030.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,9776.814286,14595.785714,10000.814286
JUNGLE,10475.100000,12700.300000,9705.042857
MIDDLE,11315.314286,14644.771429,9389.142857
TOP,11706.500000,14088.157143,9298.914286
UTILITY,8396.857143,9218.814286,7128.600000


#### 30분 ~ 40분 사이의 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인하기

In [60]:
# 30~40분 사이의 게임에서는 한타, 오브젝트 관리에 대해서 가장 중요한 시기이다. 또한 스플릿 등등 라인 관리에 대해서도 중요하다. 
# 게임의 딜량에 대해서는 원딜, 미드, 탑에 의존하는 경우가 높아진다. 
# 챔피언 경험치에 대해서는 원딜의 솔라인으로 인해서 정글과 비슷해지며 탑의 경우에는 텔포를 바탕으로 스플릿을 진행하여 여전히 높다. 
# champEXP MIDDLE 95%, TOP 90%, UTILITY 125% 
# totalDamageDealtToCHamp BOTTOM 85%, MIDDLE 85%, TOP 95%, UTILITY 150% 
# goldEARNED UTILITY 140% 
game_played3040.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,15839.681818,26685.727273,14990.363636
JUNGLE,15887.704545,21923.068182,13798.409091
MIDDLE,16872.159091,26271.818182,14049.386364
TOP,17734.613636,24110.363636,13828.840909
UTILITY,12712.522727,15194.386364,9735.681818


#### 40분 이상 종료된 게임의 라인별 딜량, 골드 획득, 경험치 분석

In [61]:
# 40분 이상의 경우에는 원딜, 미드의 딜량이 가장 중요하며 탑의 스플릿, 바론, 장로와 같은 오브젝트 싸움 한번에 게임이 끝난다.
# 원딜의 경우에는 탑, 정글, 미드의 집중으로 인해서 폭사하는 경우도 생기며 끝까지 살아서 딜을 폭발시키기도 한다.
# 최대 경험치에 이르는 경우도 많다. 
# champExp UTILITY 120%
# total DamageDealtToChamp JUNGLE 110%, MIDDLE 90%, UTILITY 110%
# goldEarned BOTTOM 90%, MIDDLE 90%, UTILITY 125%
game_played40up.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,21554.166667,34018.000000,18585.833333
JUNGLE,21350.333333,29677.666667,16976.333333
MIDDLE,22528.333333,38228.666667,18366.666667
TOP,22621.166667,34041.500000,17582.000000
UTILITY,18144.333333,31151.166667,13641.000000


## 분석 데이터프레임 수정하기

In [62]:
game_df[(game_df['timePlayed'] < 2000) & (game_df['teamPosition']== 'TOP')]

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
30,B6f-hh1my-rb1rAAAQTgeJGKVcSXZxZpVPRdWWsUvVeVDO...,TOP,TOP,Nilah,6025.0,4.0,12.0,1.0,3.0,0.0,...,29.0,4026.0,296.781612,88.0,7.0,1.0,False,True,False,1355.0
35,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,7597.0,4.0,12.0,3.0,1.0,1.0,...,24.0,5326.0,392.643939,96.0,8.0,2.0,False,True,False,1355.0
100,9yf8Rs_z-wYgmGrOEt3c5Dxt5vxXciWjSutOB3cJe_X5_b...,TOP,TOP,Jax,6558.0,12.0,4.0,2.0,7.0,0.0,...,30.0,5036.0,320.051088,80.0,1.0,0.0,False,True,False,1573.0
105,qgPQotn5woPPNGVWHTMKkNsxSO15eUKgKArQWn2D06bxpW...,TOP,TOP,Darius,8809.0,4.0,6.0,8.0,2.0,0.0,...,31.0,7298.0,463.812662,117.0,45.0,3.0,False,True,False,1573.0
130,yKZnzbQCJcT-VAKRYJSFdo_wn8SGREc6G4yiCOdiRRkJEH...,TOP,TOP,Darius,6800.0,6.0,4.0,2.0,1.0,0.0,...,15.0,5289.0,347.043530,116.0,38.0,1.0,False,True,False,1523.0
135,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Akali,6456.0,4.0,12.0,0.0,2.0,2.0,...,17.0,4409.0,289.324512,87.0,0.0,1.0,False,True,False,1523.0
170,nxp0FYb8Bm2X7Is7P_8t_LytQBIIU-48MqPrBgruO4dVeX...,TOP,TOP,Kayle,10480.0,12.0,4.0,2.0,1.0,0.0,...,16.0,7990.0,409.279655,154.0,36.0,1.0,False,True,False,1952.0
175,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,8837.0,4.0,12.0,1.0,2.0,0.0,...,20.0,5923.0,303.391759,118.0,7.0,1.0,False,True,False,1952.0
640,D58qSu1hYSOnEBJs74qZ5VCt6PymDZOnoCGErGYRUz_NWz...,TOP,TOP,Soraka,7692.0,12.0,4.0,2.0,4.0,7.0,...,17.0,5395.0,285.634450,68.0,7.0,2.0,False,True,False,1888.0
645,5QiWucuMelUIjTolOPSi2MLBaq1br8iHA6cEbw84cKQAww...,TOP,TOP,Tristana,6979.0,4.0,14.0,4.0,4.0,0.0,...,15.0,6811.0,360.623341,100.0,46.0,2.0,False,True,False,1888.0


In [65]:
game_df.loc[11]

puuid                           CMwcORn3va6QDUCuWIdAFVOhl_oMV3aR-BsVQXSGpGMZm5...
teamPosition                                                               JUNGLE
individualPosition                                                         JUNGLE
champName                                                                 Warwick
champExp                                                                   7978.0
summoner_spell1                                                               4.0
summoner_spell2                                                              11.0
kills                                                                         4.0
deaths                                                                       13.0
assists                                                                       4.0
damageDealtToBuildings                                                        0.0
totalDamageDealtToChamp                                                   12768.0
damagePerMin    

In [22]:
game_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   puuid                         650 non-null    object 
 1   teamPosition                  650 non-null    object 
 2   individualPosition            650 non-null    object 
 3   champName                     650 non-null    object 
 4   champExp                      650 non-null    float64
 5   summoner_spell1               650 non-null    float64
 6   summoner_spell2               650 non-null    float64
 7   kills                         650 non-null    float64
 8   deaths                        650 non-null    float64
 9   assists                       650 non-null    float64
 10  damageDealtToBuildings        650 non-null    float64
 11  totalDamageDealtToChamp       650 non-null    float64
 12  damagePerMin                  650 non-null    float64
 13  teamD

In [23]:
game_df.describe()

,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,damageDealtToBuildings,totalDamageDealtToChamp,damagePerMin,teamDamagePer(%),killParticipation(%),totalDamageTaken,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,timePlayed
count,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000,650.000000
mean,12270.356923,8.841538,5.996923,5.395385,5.409231,7.249231,2510.470769,16589.181538,565.594602,20.000000,46.331783,21356.984615,20.003077,10524.729231,372.138008,138.806154,24.847692,1.506154,2807.261538
std,4394.870493,4.404907,3.567529,4.471686,3.229569,5.105908,2872.260880,10357.818890,271.772810,7.773766,16.098798,10950.290099,5.872292,3840.010878,87.982566,73.361208,29.200513,0.966683,719.503773
min,3038.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1243.000000,70.375358,3.000000,0.000000,1419.000000,5.000000,2442.000000,169.301926,1.000000,0.000000,0.000000,1355.000000
25%,8960.000000,4.000000,4.000000,2.000000,3.000000,3.000000,353.500000,9339.250000,370.367133,14.000000,36.363636,13481.250000,16.000000,7696.750000,307.888053,92.250000,4.000000,1.000000,2272.000000
50%,11575.500000,11.000000,4.000000,4.000000,5.000000,6.000000,1620.500000,14036.000000,523.887225,20.000000,47.222222,19185.500000,20.000000,9968.000000,364.155865,141.000000,13.000000,1.000000,2615.000000
75%,15191.750000,14.000000,7.000000,7.000000,7.000000,10.000000,3619.750000,21570.250000,725.418916,25.000000,56.493868,27221.500000,24.000000,12813.750000,433.442378,188.000000,36.937500,2.000000,3315.000000
max,26517.000000,21.000000,21.000000,26.000000,17.000000,27.000000,25514.000000,66793.000000,1768.984191,44.000000,100.000000,80835.000000,38.000000,23339.000000,700.094282,370.000000,168.000000,6.000000,4815.000000


In [35]:
game_df.loc[10]['champExp']

10612.0

In [64]:
# 위에서 분석한 시간대별 골드획득, 적에게 가한 피해량, 경험치 획득 지표를 통해서 값 변경해주기
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for i in range(len(game_df)):
    if game_df.loc[i, 'timePlayed'] < 2000:

        if game_df.loc[i, 'teamPosition'] == 'TOP':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.85
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.9
            
        elif game_df.loc[i, 'teamPosition'] == 'JUNGLE':
             game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.05
            
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.9
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.75
            
        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.1

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.3
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.55
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.35
            
        else:
            pass
    elif (2000 <= game_df.loc[i, 'timePlayed']) & (game_df.loc[i, 'timePlayed'] < 3000):      
        if game_df.loc[i,  'teamPosition'] == 'JUNGLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.05
        
        elif game_df.loc[i, 'teamPosition']== 'BOTTOM':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.1


        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.25
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.4
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.3
            
        else:
            pass
    
    elif (3000 <= game_df.loc[i,'timePlayed']) & (game_df.loc[i, 'timePlayed'] < 4000):

        if game_df.loc[i, 'teamPosition'] == 'TOP':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.9
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *0.85
        
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.95
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.85

        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.85

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.25
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.5
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.4 
    elif game_df.loc[i, 'timePlayed'] >=4000:
        
        if game_df.loc[i, 'teamPosition'] == 'JUNGGLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *1.1
        
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.9
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 0.9
        
        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 0.9

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.2
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *1.1
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.25

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27